<a href="https://colab.research.google.com/github/Borwec/ida_25_26/blob/main/lab4/lab4_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from transformers import pipeline
import re
import string
import pandas as pd
import numpy as np
import gc
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


Task 3a

In [ ]:
#https://www.kaggle.com/datasets/sayankr007/cyber-bullying-data-for-multi-label-classification

ds_url = "https://drive.google.com/uc?id=1ayAK_Z3ZJOq2Fqp9ATyt694ha9iJ1sYM"
df = pd.read_csv(ds_url, usecols=["comment", "label"])
df.head()

,comment,label
0,0 u0 lmao wow fuck you too 😂 😂,normal
1,1 0 th floor maybe wow cnn with the fakenews t...,offensive
2,1 0 yrs <number> white women raped by niggers ...,hatespeech
3,1 2 h ago ching chong accepted your friend req...,offensive
4,1 8 th century mayhem and lawlessness had noth...,normal


In [ ]:
def prep_text(text):
  text = text.lower()
  text = re.sub(r"[^a-z<>]", " ", text)
  text = re.sub(r"\s{2,}", " ", text)
  return text

for i in range(len(df["comment"])):
  df.loc[i, "comment"] = prep_text(df["comment"][i])

df.head()

,comment,label
0,u lmao wow fuck you too,normal
1,th floor maybe wow cnn with the fakenews the ...,offensive
2,yrs <number> white women raped by niggers <nu...,hatespeech
3,h ago ching chong accepted your friend request,offensive
4,th century mayhem and lawlessness had nothing...,normal


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df["comment"], df["label"], test_size=0.25, random_state=127)
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=0.05)

tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)

rf = RandomForestClassifier(random_state=127, n_jobs=-1)
rf.fit(tfidf_train, y_train)
pred = rf.predict(tfidf_test)

print(classification_report(y_test,pred))
print()
print(confusion_matrix(y_test,pred))
print()
print(accuracy_score(y_test,pred))

              precision    recall  f1-score   support

  hatespeech       0.50      0.51      0.50      1600
      normal       0.51      0.63      0.56      1958
   offensive       0.35      0.23      0.28      1470

    accuracy                           0.47      5028
   macro avg       0.45      0.46      0.45      5028
weighted avg       0.46      0.47      0.46      5028


[[ 814  495  291]
 [ 372 1226  360]
 [ 441  686  343]]

0.4739459029435163


In [ ]:
clf = pipeline("text-classification", model="Hate-speech-CNERG/bert-base-uncased-hatexplain")
clf_test = y_test.map({"normal":"normal", "offensive":"offensive", "hatespeech":"hate speech"})
pred = []
for i in range(len(X_test)):
  pred.append(clf(X_test.iloc[i])[0]["label"])

print(classification_report(clf_test,pred))
print()
print(confusion_matrix(clf_test,pred))
print()
print(accuracy_score(clf_test,pred))

              precision    recall  f1-score   support

 hate speech       0.78      0.73      0.75      1600
      normal       0.75      0.73      0.74      1958
   offensive       0.56      0.61      0.58      1470

    accuracy                           0.70      5028
   macro avg       0.70      0.69      0.69      5028
weighted avg       0.70      0.70      0.70      5028


[[1164  118  318]
 [ 119 1436  403]
 [ 214  355  901]]

0.6963007159904535


In [ ]:
clf = pipeline("text-classification", model="s-nlp/roberta_toxicity_classifier")
clf_test = y_test.map({"normal":"neutral", "offensive":"toxic", "hatespeech":"toxic"})
pred = []
for i in range(len(X_test)):
  pred.append(clf(X_test.iloc[i])[0]["label"])

print(classification_report(clf_test,pred))
print()
print(confusion_matrix(clf_test,pred))
print()
print(accuracy_score(clf_test,pred))

Some weights of the model checkpoint at s-nlp/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


              precision    recall  f1-score   support

     neutral       0.55      0.60      0.58      1958
       toxic       0.73      0.69      0.71      3070

    accuracy                           0.65      5028
   macro avg       0.64      0.65      0.64      5028
weighted avg       0.66      0.65      0.66      5028


[[1177  781]
 [ 954 2116]]

0.6549323786793954


In [ ]:
del(clf)
gc.collect()
torch.cuda.empty_cache()

Task 3b

In [ ]:
ua_jp = pipeline("translation", model="facebook/nllb-200-distilled-600M")

Device set to use cuda:0


In [ ]:
samples = [
    "Спробуємо перефразування тексту шляхом використання надійного методу перекладу на якусь мову і назад",
    "Було проведено роботу з експериметнування над моделями",
    "Результати непогані, але отриманий переклад потребує доопрацювання"
]

translations = []
for s in samples:
  text_ua_jp = ua_jp(s, max_length=100, src_lang="ukr_Cyrl", tgt_lang="jpn_Jpan")[0]["translation_text"]
  text_jp_ua = ua_jp(text_ua_jp, max_length=100, src_lang="jpn_Jpan", tgt_lang="ukr_Cyrl")[0]["translation_text"]
  translations.append(text_jp_ua)

for s, t in zip(samples, translations):
  print(f"Вихідний текст: \t{s}\nПерекладений текст: \t{t}\n")

Вихідний текст: 	Спробуємо перефразування тексту шляхом використання надійного методу перекладу на якусь мову і назад
Перекладений текст: 	Давайте перепишемо вислів, використовуючи надійний метод перекладу.

Вихідний текст: 	Було проведено роботу з експериметнування над моделями
Перекладений текст: 	Ми провели експерименти з моделью.

Вихідний текст: 	Результати непогані, але отриманий переклад потребує доопрацювання
Перекладений текст: 	Це добре, але переклад повинен бути поліпшений.



In [26]:
del(ua_jp)
gc.collect()
torch.cuda.empty_cache()

In [40]:
summ = pipeline("summarization", "SGaleshchuk/t5-large-ua-news")

text = """За даними Укренерго, 30 грудня у більшості регіонів України будуть діяти графіки погодинних відключень та графіки обмеження потужності (для промислових споживачів).

Причиною запровадження заходів обмеження стали наслідки російських масованих ракетно-дронових атак на об’єкти енергетичної інфраструктури."""
print(summ(text, min_length=3, max_length = 64)[0]["summary_text"])

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda:0


У більшості регіонів України 30 грудня будуть діяти графіки погодинних відключень та обмеження потужності (для промислових споживачів). 


In [41]:
del(summ)
gc.collect()
torch.cuda.empty_cache()

In [7]:
zsc = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7")

text = "У більшості регіонів України 30 грудня будуть діяти графіки погодинних відключень та обмеження потужності (для промислових споживачів)."
candidates = ["спорт", "освіта", "новини"]

zsc(text, candidate_labels=candidates)

Device set to use cpu


{'sequence': 'У більшості регіонів України 30 грудня будуть діяти графіки погодинних відключень та обмеження потужності (для промислових споживачів).',
 'labels': ['новини', 'освіта', 'спорт'],
 'scores': [0.6878549456596375, 0.26357802748680115, 0.04856695234775543]}

In [5]:
del(zsc)
gc.collect()
torch.cuda.empty_cache()